# Feedback to Policy — early project-discovery check

**Illustrative user need:** Policy officers want to find and compare EU-funded research projects relevant to their work, and reuse trustworthy project information for policy discussions.

The policy theme would **normally be agreed with users**. For demonstration, the editable search below uses **digital education**. It is **not** a validated requirement, a complete CORDIS search or a policy analysis.

**Simple workflow:** define an example question → inspect selected data → check quality → run an explainable keyword baseline → review matches **and omissions** → export files for user feedback.

This supports Sections 2–4 of the accompanying written note. No AI model or non-public data is used.

## 1. Define the illustrative policy question

In a real engagement I would first confirm the intended **policy decision**, obtain examples of relevant/irrelevant projects from users, and check **existing F2P/CORDIS tools**. Those activities cannot be simulated as if they happened.

For this demonstration, change `policy_question`, `search_terms` and `minimum_terms` to explore another policy topic. A small hand-curated dataset cannot validate search effectiveness.

In [1]:
policy_question = "Which EU-funded projects may relate to digital education?"  # Illustrative only
search_terms = ["digital", "education", "learning"]  # To be refined with policy users
minimum_terms = 2  # Simple screening rule, not a relevance score
print("Question:", policy_question)
print("Terms:", ", ".join(search_terms))

Question: Which EU-funded projects may relate to digital education?
Terms: digital, education, learning


## 2. Load the data and assess its suitability

The input has **12 public CORDIS-linked project records** spanning different areas (energy, education and health). It is deliberately mixed rather than selected around one predetermined topic. Official project identifiers/titles are paired with **short analyst-written descriptions** and illustrative domain labels.

These descriptions are **not** official abstracts, and the list is **not** an official CORDIS export or representative sample. The data would need refreshing and validation for real use.

In [2]:
from pathlib import Path
import pandas as pd

# Locate project files from the repository root or from notebooks/.
root = Path.cwd()
if not (root / "data" / "sample_projects.csv").exists():
    root = root.parent
projects = pd.read_csv(root / "data" / "sample_projects.csv", dtype=str).fillna("")
print(f"Records loaded: {len(projects)}")
display(projects[["acronym", "project_title", "example_domain"]].head())

Records loaded: 12


,acronym,project_title,example_domain
0,Solhyd,Worldwide Unique Solar Hydrogen Modules: Built...,Energy
1,X-STREAM,Power-to-X: STREAMing Hydrogen from 3-Band Sol...,Energy
2,SUNKID,Single jUNction perovsKIte solar cell powereD ...,Energy
3,BioInspired_SolarH2,Engineering Bio-Inspired Systems for the Conve...,Energy
4,EMPOWER,Design and evaluation of technological support...,Education


## 3. Check basic metadata quality

Before trusting a shortlist, check whether records can be traced to a source, whether essential fields are present and whether grant identifiers are duplicated. Checking URL **format** does not prove that its contents are correct or current.

In [3]:
required = ["grant_id", "project_title", "short_description", "cordis_url"]
quality = pd.DataFrame({
    "check": ["Missing essential values", "Duplicate grant IDs", "Unexpected CORDIS link format"],
    "count": [
        int(projects[required].eq("").sum().sum()),
        int(projects["grant_id"].duplicated().sum()),
        int((~projects["cordis_url"].str.startswith("https://cordis.europa.eu/project/id/")).sum()),
    ],
})
display(quality)

,check,count
0,Missing essential values,0
1,Duplicate grant IDs,0
2,Unexpected CORDIS link format,0


## 4. Test a transparent search baseline

Start with a straightforward rule instead of an AI model. A record is a **candidate** if its title or prepared description contains at least `minimum_terms` of the listed terms. The result includes the terms that triggered selection.

This can create **false matches** and **miss relevant projects** that use different wording. It does not represent a validated classification or measure the performance of CORDIS search.

In [4]:
projects["search_text"] = (
    projects["project_title"] + " " + projects["short_description"]
).str.lower()
projects["matched_terms"] = projects["search_text"].apply(
    lambda text: ", ".join(term for term in search_terms if term.lower() in text)
)
projects["terms_found"] = projects["matched_terms"].apply(
    lambda text: len([term for term in text.split(", ") if term])
)
projects["candidate"] = projects["terms_found"] >= minimum_terms

print(f"Provisional candidates: {int(projects['candidate'].sum())} of {len(projects)} sample records")
display(projects[["acronym", "candidate", "matched_terms", "cordis_url"]])

Provisional candidates: 4 of 12 sample records


,acronym,candidate,matched_terms,cordis_url
0,Solhyd,False,,https://cordis.europa.eu/project/id/101297101
1,X-STREAM,False,,https://cordis.europa.eu/project/id/101124803
2,SUNKID,False,,https://cordis.europa.eu/project/id/101209110
3,BioInspired_SolarH2,False,,https://cordis.europa.eu/project/id/805524
4,EMPOWER,True,"digital, education",https://cordis.europa.eu/project/id/101060918
5,e-DIPLOMA,True,"digital, education, learning",https://cordis.europa.eu/project/id/101061424
6,EIHEUDL,True,"education, learning",https://cordis.europa.eu/project/id/101244212
7,SandCASLE,True,"education, learning",https://cordis.europa.eu/project/id/101310819
8,THCS,False,,https://cordis.europa.eu/project/id/101095654
9,EDiHTA,False,digital,https://cordis.europa.eu/project/id/101136424


## 5. Compare the illustrative subset and prepare user review

A quick grouping can help show what the **selected sample** contains. The `example_domain` labels were assigned for this exercise; the counts are **not statistics about EU-funded research**. Both retrieved and excluded records must be reviewed to understand false matches and possible omissions.

In [5]:
display(projects.groupby(["example_domain", "candidate"]).size().unstack(fill_value=0))

output_dir = root / "outputs"
output_dir.mkdir(exist_ok=True)
review_columns = ["grant_id", "acronym", "project_title", "example_domain", "cordis_url", "candidate", "matched_terms"]
review = projects[review_columns].copy()
review["user_relevance"] = ""      # To be completed by policy officers
review["user_comment"] = ""        # Questions / missing context / possible false negatives
review.to_csv(output_dir / "all_projects_for_review.csv", index=False)
review.loc[review["candidate"]].to_csv(output_dir / "candidate_shortlist.csv", index=False)
quality.to_csv(output_dir / "data_quality_checks.csv", index=False)
print("Review files exported to outputs/ (no user validation claimed).")

candidate,False,True
example_domain,,
Education,0,4
Energy,4,0
Health,4,0


Review files exported to outputs/ (no user validation claimed).


## 6. What would be validated with users?

1. **Scope:** Is the policy question and selected project universe appropriate? Which projects are known to be in/out of scope?
2. **Relevance:** Which candidates are genuinely useful, and which excluded records were missed?
3. **Interpretation and traceability:** Are the title, source link and project description enough for a briefing? Do users need *actual research findings*, which would require a **separate validation step**?
4. **Decision:** If existing F2P/CORDIS features already satisfy the need, reuse them; otherwise refine the baseline, test on authorised representative data, or stop.

**No precision/recall or time-saving claim is made.** Those measures require user-labelled examples and a realistic benchmark. Semantic retrieval or LLM assistance should be considered only if the baseline is insufficient and a controlled comparison supports additional value.